<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Tiny-Transformer/blob/main/tiny_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# *1.* Tokenization <br>

A Tokenization é a primeira fase na construção de uma arquitetura Transformers. <br> Consiste em transformar o dataset em palavras mais curtas (tokens) mas sem perder o seu significado.




In [3]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file ("/Os Lusiadas Tokenizer.json") #Tokenizer criado para este dataset, não é comum. Modeos atuais utilizam já tokenizers construídos.

with open ("/Os Lusiadas.txt", "r", encoding = "utf-8") as file:
  doc = file.read()

dados_tokenizer = tokenizer.encode (doc) #Encode encarrega-se de converter o dataset para Tokens. Todos os Tokens têm o seu ID

print (dados_tokenizer.tokens[:100])
print (dados_tokenizer.ids[:100])
print (len(dados_tokenizer.tokens))

['O', 'S', 'L', 'U', 'S', 'Í', 'A', 'D', 'A', 'S', 'Lu', 'ís', 'de', 'Cam', 'ões', 'Canto', 'I', 'As', 'armas', 'e', 'os', 'Bar', 'ões', 'assinala', 'dos', 'Que', 'da', 'Oci', 'dent', 'al', 'praia', 'Lusitana', 'Por', 'mares', 'nunca', 'de', 'antes', 'navega', 'dos', 'Passa', 'ram', 'ainda', 'além', 'da', 'Ta', 'probana', ',', 'Em', 'perigos', 'e', 'guerras', 'esforça', 'dos', 'Mais', 'do', 'que', 'prome', 'tia', 'a', 'força', 'humana', ',', 'E', 'entre', 'gente', 'remota', 'edifica', 'ram', 'Novo', 'Reino', ',', 'que', 'tanto', 'sublima', 'ram', ';', 'E', 'também', 'as', 'memórias', 'gloriosas', 'Daqueles', 'Reis', 'que', 'foram', 'dila', 'tando', 'A', 'Fé', ',', 'o', 'Império', ',', 'e', 'as', 'terras', 'vi', 'cios', 'as', 'De']
[27, 31, 24, 33, 31, 70, 14, 17, 14, 31, 3537, 465, 100, 1187, 340, 1823, 22, 239, 526, 43, 93, 2172, 340, 2680, 133, 125, 104, 2418, 2234, 110, 1571, 1221, 171, 919, 453, 100, 460, 1576, 133, 2386, 191, 3565, 4435, 104, 2189, 4670, 8, 312, 852, 43, 1655, 246

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as FUNCTION

tensor = torch.tensor(dados_tokenizer.ids, dtype = torch.long) #Colocação dos IDS num tensor de dimensão [1, 0]
print (tensor)

#Divisão dos Dados
n = int (0.9 * len(tensor))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

tensor([  27,   31,   24,  ...,  147, 2255,   10])


### *1.2* Context Length e Batch Size <br>
É completamente impossível alimentar uma arquitetura Transformer com os dados logo todos de uma vez. Computacionalmente é impossível. <br> Para resolver isto, vamos alimentando o modelo com excertos de Tokens. <br>
**Context Lenght** é uma métrica que define quantos Tokens o modelo vê numa sequência. <br>
**Batch Size** é quantas sequências o modelo vê ao mesmo tempo.

In [5]:
context_len = 8
batch_size = 4

torch.manual_seed(1000)

def get_batch ():
  position = torch.randint (0, len(dados_treino) - context_len, (batch_size, ))
  x = torch.stack ([dados_treino [i : i + context_len] for i in position])
  y = torch.stack ([dados_treino [i + 1 : i + context_len + 1] for i in position])

  return x, y

inputs, targets = get_batch()

print ('inputs')
print (inputs.shape) #torch.Size([4, 8])
print (inputs)

print ('targets')
print (targets.shape) #torch.Size([4, 8])
print (targets)


inputs
torch.Size([4, 8])
tensor([[  95, 1810,  589, 2717,  210,   39,  520, 2861],
        [3832,  101, 1956, 2646,  191,   11,   63,   71],
        [4665,   12,  414, 3292,  104,  189,  100, 1956],
        [1638,  729,    8, 1982,  192, 1935,   43,  192]])
targets
torch.Size([4, 8])
tensor([[1810,  589, 2717,  210,   39,  520, 2861,   43],
        [ 101, 1956, 2646,  191,   11,   63,   71,  156],
        [  12,  414, 3292,  104,  189,  100, 1956,   39],
        [ 729,    8, 1982,  192, 1935,   43,  192, 1219]])
